# ml_experiments.ipynb
## Experimentos de Aprendizaje Automático
### TFG — Análisis de biomarcadores acústicos en ELA

**Autor:** Jakub Wysocki 

---

### Diseño experimental (4 escenarios × 2 etiquetas = 8 experimentos)

| # | Escenario | Etiquetas | Relevancia clínica |
|---|-----------|-----------|--------------------|
| 1 | ELA_bulbar vs ELA_no_bulbar | Clínico | Discriminación de fenotipo dentro de la patología |
| 2 | ELA_bulbar vs Control | Clínico | Detección de afectación bulbar manifiesta |
| 3 | ELA_no_bulbar vs Control | Clínico | Detección precoz sin afectación clínica |
| 4 | Todos ELA vs Control | Clínico | Baseline diagnóstico general |
| 5–8 | Mismos escenarios | Reetiquetado máquina (S4VM) | Evaluación del reetiquetado computacional |

### Modelos evaluados
LogisticRegression (baseline) · RandomForest · XGBoost · LightGBM · CatBoost · SVM

### Decisiones metodológicas (justificadas por el EDA)
| Decisión | Justificación EDA |
|----------|-------------------|
| `RobustScaler` | 44 features con ≥5 outliers (IQR) |
| `class_weight='balanced'` | Desbalanceo: ELA_bulbar=14, ELA_no_bulbar=31, Control=18 |
| CV estratificada k=5 | n=63, clase mínima=14 |
| Sin reducción de dimensionalidad | PCA solo explica 21.4% en 2D; 44 componentes para 95% |
| Optuna (búsqueda bayesiana) | Optimización eficiente del espacio de hiperparámetros |
| SHAP | Explicabilidad de las predicciones del mejor modelo |

## 0. Importaciones y configuración

In [ ]:
import sys
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ML
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_curve, auc, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Optimización e interpretabilidad
import optuna
import shap
import mlflow
import mlflow.sklearn

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

PROJECT_ROOT  = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR   = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_DIR    = PROJECT_ROOT / 'mlruns'

RANDOM_STATE = 42
N_FOLDS      = 5
N_OPTUNA_TRIALS = 50  # Trials de Optuna por modelo

CLASS_PALETTE = {
    'ELA_bulbar':    '#E05C4B',
    'ELA_no_bulbar': '#F4A236',
    'Control':       '#4B9CD3',
}

print('✓ Importaciones completadas')
print(f'  Proyecto: {PROJECT_ROOT}')

## 1. Carga del dataset final

In [ ]:
df = pd.read_csv(PROCESSED_DIR / 'dataset_final.csv')

META_COLS    = ['subject_id', 'genero', 'label_clinico', 'label_maquina']
FEATURE_COLS = [c for c in df.columns if c not in META_COLS]

print(f'Dataset: {df.shape}')
print(f'Features acústicas: {len(FEATURE_COLS)} (5 vocales × 40 biomarcadores)')
print(f'\nlabel_clinico:\n{df["label_clinico"].value_counts().to_string()}')
print(f'\nlabel_maquina:\n{df["label_maquina"].value_counts().to_string()}')

# Verificaciones de integridad
assert df[FEATURE_COLS].isnull().sum().sum() == 0, 'Hay nulos en features'
assert df.duplicated().sum() == 0, 'Hay filas duplicadas'
print('\n✓ Dataset sin nulos ni duplicados')

## 2. Definición de escenarios, modelos y funciones auxiliares

In [ ]:
# ---------------------------------------------------------------------------
# Escenarios: (nombre, clase_positiva, clase_negativa)
# ---------------------------------------------------------------------------
SCENARIOS = [
    ('Bulbar_vs_NoBulbar',  'ELA_bulbar',    'ELA_no_bulbar'),
    ('Bulbar_vs_Control',   'ELA_bulbar',    'Control'),
    ('NoBulbar_vs_Control', 'ELA_no_bulbar', 'Control'),
    ('AllELA_vs_Control',   'ELA',           'Control'),
]

LABEL_SYSTEMS = ['label_clinico', 'label_maquina']

# ---------------------------------------------------------------------------
# Modelos base (sin optimización de hiperparámetros)
# Todos con RobustScaler en el pipeline y class_weight='balanced'
# ---------------------------------------------------------------------------
def get_base_models() -> dict[str, Pipeline]:
    return {
        'LogisticRegression': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', LogisticRegression(
                class_weight='balanced', max_iter=1000,
                random_state=RANDOM_STATE,
            )),
        ]),
        'RandomForest': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', RandomForestClassifier(
                n_estimators=200, class_weight='balanced',
                random_state=RANDOM_STATE, n_jobs=-1,
            )),
        ]),
        'XGBoost': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', XGBClassifier(
                n_estimators=200, max_depth=4, learning_rate=0.05,
                eval_metric='logloss', random_state=RANDOM_STATE, verbosity=0,
            )),
        ]),
        'LightGBM': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', LGBMClassifier(
                n_estimators=200, max_depth=4, learning_rate=0.05,
                class_weight='balanced', random_state=RANDOM_STATE, verbose=-1,
            )),
        ]),
        'CatBoost': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', CatBoostClassifier(
                iterations=200, depth=4, learning_rate=0.05,
                auto_class_weights='Balanced',
                random_seed=RANDOM_STATE, verbose=0,
            )),
        ]),
        'SVM': Pipeline([
            ('scaler', RobustScaler()),
            ('clf', SVC(
                kernel='rbf', class_weight='balanced',
                probability=True, random_state=RANDOM_STATE,
            )),
        ]),
    }

print(f'Modelos definidos: {list(get_base_models().keys())}')
print(f'Escenarios: {len(SCENARIOS)}')
print(f'Sistemas de etiquetas: {len(LABEL_SYSTEMS)}')
print(f'Total experimentos base: {len(SCENARIOS)*len(LABEL_SYSTEMS)*len(get_base_models())} runs')

In [ ]:
# ---------------------------------------------------------------------------
# Funciones auxiliares
# ---------------------------------------------------------------------------

def prepare_binary_dataset(
    df: pd.DataFrame,
    label_col: str,
    pos_class: str,
    neg_class: str,
) -> tuple[np.ndarray, np.ndarray, dict]:
    """
    Prepara X, y para un experimento binario.
    Para 'AllELA_vs_Control' agrupa ELA_bulbar + ELA_no_bulbar.
    """
    df_exp = df.copy()

    if pos_class == 'ELA':
        mask = df_exp[label_col].isin(['ELA_bulbar', 'ELA_no_bulbar', 'Control'])
        df_exp = df_exp[mask].copy()
        df_exp['_y'] = df_exp[label_col].apply(
            lambda x: 1 if x in ('ELA_bulbar', 'ELA_no_bulbar') else 0
        )
    else:
        mask = df_exp[label_col].isin([pos_class, neg_class])
        df_exp = df_exp[mask].copy()
        df_exp['_y'] = (df_exp[label_col] == pos_class).astype(int)

    X = df_exp[FEATURE_COLS].values.astype(float)
    y = df_exp['_y'].values

    n_pos = int(y.sum())
    n_neg = int((y == 0).sum())
    return X, y, {'n': len(y), 'n_pos': n_pos, 'n_neg': n_neg}


def compute_cv_metrics(
    pipeline: Pipeline,
    X: np.ndarray,
    y: np.ndarray,
) -> dict:
    """
    Calcula AUC-ROC, F1-macro, Accuracy, Recall, Especificidad
    mediante validación cruzada estratificada k=5.
    """
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    scoring = {
        'roc_auc':  'roc_auc',
        'f1_macro': 'f1_macro',
        'accuracy': 'accuracy',
        'recall':   'recall',
    }
    cv_res = cross_validate(
        pipeline, X, y, cv=cv, scoring=scoring,
        return_train_score=False, n_jobs=-1,
    )

    # Especificidad calculada manualmente (no disponible en sklearn scoring)
    spec_scores = []
    for train_idx, test_idx in cv.split(X, y):
        pipeline.fit(X[train_idx], y[train_idx])
        y_pred = pipeline.predict(X[test_idx])
        cm = confusion_matrix(y[test_idx], y_pred)
        tn, fp = (cm[0, 0], cm[0, 1]) if cm.shape == (2, 2) else (0, 0)
        spec_scores.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

    metrics = {}
    for name, key in [('auc_roc', 'test_roc_auc'), ('f1_macro', 'test_f1_macro'),
                      ('accuracy', 'test_accuracy'), ('recall', 'test_recall')]:
        arr = cv_res[key]
        metrics[f'{name}_mean'] = float(np.mean(arr))
        metrics[f'{name}_std']  = float(np.std(arr))

    metrics['specificity_mean'] = float(np.mean(spec_scores))
    metrics['specificity_std']  = float(np.std(spec_scores))
    return metrics


print('✓ Funciones auxiliares definidas')

## 3. Experimentos base — todos los modelos, todos los escenarios

In [ ]:
mlflow.set_tracking_uri(MLFLOW_DIR.as_uri())
mlflow.set_experiment('ELA_Acoustic_Biomarkers')

results: list[dict] = []

for label_system in LABEL_SYSTEMS:
    label_tag = label_system.replace('label_', '')

    for scen_name, pos_cls, neg_cls in SCENARIOS:
        X, y, info = prepare_binary_dataset(df, label_system, pos_cls, neg_cls)
        exp_id = f'{scen_name}__{label_tag}'

        print(f'\n[{exp_id}]  n={info["n"]}  pos={info["n_pos"]} neg={info["n_neg"]}')

        if info['n_pos'] < N_FOLDS or info['n_neg'] < N_FOLDS:
            print('  ⚠ Clase demasiado pequeña para CV. Omitido.')
            continue

        for model_name, pipeline in get_base_models().items():
            with mlflow.start_run(run_name=f'{exp_id}__{model_name}'):
                mlflow.log_params({
                    'experiment': exp_id, 'model': model_name,
                    'label_system': label_tag, 'scenario': scen_name,
                    'pos_class': pos_cls, 'neg_class': neg_cls,
                    'n': info['n'], 'n_pos': info['n_pos'], 'n_neg': info['n_neg'],
                    'cv_folds': N_FOLDS, 'n_features': len(FEATURE_COLS),
                    'random_state': RANDOM_STATE,
                })

                metrics = compute_cv_metrics(pipeline, X, y)
                mlflow.log_metrics(metrics)

                row = {'experiment': exp_id, 'label_system': label_tag,
                       'scenario': scen_name, 'model': model_name,
                       'phase': 'base', **info, **metrics}
                results.append(row)

                print(
                    f'  {model_name:20s}  '
                    f'AUC={metrics["auc_roc_mean"]:.3f}±{metrics["auc_roc_std"]:.3f}  '
                    f'F1={metrics["f1_macro_mean"]:.3f}  '
                    f'Recall={metrics["recall_mean"]:.3f}  '
                    f'Spec={metrics["specificity_mean"]:.3f}'
                )

results_df = pd.DataFrame(results)
results_df.to_csv(PROCESSED_DIR / 'ml_results_base.csv', index=False)
print(f'\n✓ Experimentos base completados: {len(results_df)} runs')

## 4. Tabla de resultados base

In [ ]:
display_cols = ['experiment', 'model', 'auc_roc_mean', 'auc_roc_std',
                'f1_macro_mean', 'recall_mean', 'specificity_mean', 'accuracy_mean']

results_df[display_cols].sort_values(
    ['experiment', 'auc_roc_mean'], ascending=[True, False]
).round(3).style.background_gradient(
    subset=['auc_roc_mean', 'f1_macro_mean'], cmap='RdYlGn', vmin=0.4, vmax=1.0
)

## 5. Optimización de hiperparámetros con Optuna

Se aplica al **mejor modelo por escenario** según AUC-ROC base.
Optuna implementa búsqueda bayesiana (TPE) con pruning automático de trials no prometedores,
superior a grid search y random search en eficiencia (Akiba et al., 2019).

In [ ]:
def get_optuna_objective(model_name: str, X: np.ndarray, y: np.ndarray):
    """
    Devuelve la función objetivo de Optuna para el modelo indicado.
    La métrica objetivo es AUC-ROC estimada por CV interna.
    """
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    def objective(trial: optuna.Trial) -> float:
        if model_name == 'RandomForest':
            params = {
                'n_estimators':  trial.suggest_int('n_estimators', 100, 500),
                'max_depth':     trial.suggest_int('max_depth', 3, 15),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
                'max_features':  trial.suggest_categorical('max_features', ['sqrt', 'log2']),
            }
            clf = RandomForestClassifier(
                class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, **params
            )
        elif model_name == 'XGBoost':
            params = {
                'n_estimators':  trial.suggest_int('n_estimators', 50, 400),
                'max_depth':     trial.suggest_int('max_depth', 2, 8),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'subsample':     trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'reg_alpha':     trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
                'reg_lambda':    trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
            }
            clf = XGBClassifier(
                eval_metric='logloss', random_state=RANDOM_STATE, verbosity=0, **params
            )
        elif model_name == 'LightGBM':
            params = {
                'n_estimators':  trial.suggest_int('n_estimators', 50, 400),
                'max_depth':     trial.suggest_int('max_depth', 2, 8),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'num_leaves':    trial.suggest_int('num_leaves', 10, 100),
                'subsample':     trial.suggest_float('subsample', 0.5, 1.0),
            }
            clf = LGBMClassifier(
                class_weight='balanced', random_state=RANDOM_STATE, verbose=-1, **params
            )
        elif model_name == 'CatBoost':
            params = {
                'iterations':    trial.suggest_int('iterations', 50, 400),
                'depth':         trial.suggest_int('depth', 2, 8),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'l2_leaf_reg':   trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
            }
            clf = CatBoostClassifier(
                auto_class_weights='Balanced', random_seed=RANDOM_STATE, verbose=0, **params
            )
        elif model_name == 'SVM':
            params = {
                'C':     trial.suggest_float('C', 1e-3, 100.0, log=True),
                'gamma': trial.suggest_categorical('gamma', ['scale', 'auto']),
            }
            clf = SVC(
                kernel='rbf', class_weight='balanced',
                probability=True, random_state=RANDOM_STATE, **params
            )
        else:  # LogisticRegression
            params = {
                'C':     trial.suggest_float('C', 1e-3, 100.0, log=True),
                'solver': trial.suggest_categorical('solver', ['lbfgs', 'saga']),
            }
            clf = LogisticRegression(
                class_weight='balanced', max_iter=1000,
                random_state=RANDOM_STATE, **params
            )

        pipeline = Pipeline([('scaler', RobustScaler()), ('clf', clf)])
        scores   = cross_validate(
            pipeline, X, y, cv=cv, scoring='roc_auc', n_jobs=-1
        )
        return float(np.mean(scores['test_score']))

    return objective


print('✓ Función objetivo Optuna definida')

In [ ]:
# Identificar el mejor modelo por experimento para optimizar
best_per_exp = (
    results_df
    .sort_values('auc_roc_mean', ascending=False)
    .groupby('experiment', sort=False)
    .first()
    .reset_index()
)

print('=== MEJOR MODELO BASE POR EXPERIMENTO ===')
for _, row in best_per_exp.iterrows():
    print(f'  {row["experiment"]:35s}  {row["model"]:20s}  AUC={row["auc_roc_mean"]:.3f}')

In [ ]:
# Ejecutar Optuna para el mejor modelo de cada experimento
optuna_results: list[dict] = []

for _, row in best_per_exp.iterrows():
    exp_id     = row['experiment']
    model_name = row['model']
    label_sys  = f"label_{row['label_system']}"
    scen_name  = row['scenario']

    scen = next(s for s in SCENARIOS if s[0] == scen_name)
    _, pos_cls, neg_cls = scen

    X, y, info = prepare_binary_dataset(df, label_sys, pos_cls, neg_cls)

    if info['n_pos'] < N_FOLDS or info['n_neg'] < N_FOLDS:
        continue

    print(f'\nOptimizando [{exp_id}] → {model_name} ({N_OPTUNA_TRIALS} trials)...')

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(get_optuna_objective(model_name, X, y), n_trials=N_OPTUNA_TRIALS, n_jobs=1)

    best_auc = study.best_value
    best_params = study.best_params
    print(f'  Mejor AUC (Optuna): {best_auc:.4f}')
    print(f'  Mejores params: {best_params}')

    optuna_results.append({
        'experiment': exp_id,
        'model': model_name,
        'auc_base': row['auc_roc_mean'],
        'auc_optuna': best_auc,
        'delta_auc': best_auc - row['auc_roc_mean'],
        'best_params': str(best_params),
    })

    # Guardar study en MLflow
    with mlflow.start_run(run_name=f'{exp_id}__{model_name}__optuna'):
        mlflow.log_param('model', model_name)
        mlflow.log_param('n_trials', N_OPTUNA_TRIALS)
        mlflow.log_params(best_params)
        mlflow.log_metric('auc_roc_optuna', best_auc)

optuna_df = pd.DataFrame(optuna_results)
optuna_df.to_csv(PROCESSED_DIR / 'ml_results_optuna.csv', index=False)
print(f'\n✓ Optimización Optuna completada: {len(optuna_df)} experimentos')

## 6. Visualizaciones de resultados

In [ ]:
# Heatmap: AUC-ROC base — todos los modelos × todos los experimentos
pivot = results_df.pivot_table(index='experiment', columns='model', values='auc_roc_mean')

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    pivot, annot=True, fmt='.3f', cmap='RdYlGn',
    vmin=0.4, vmax=1.0, linewidths=0.5, linecolor='white',
    ax=ax, cbar_kws={'label': 'AUC-ROC (media CV-5)'},
)
ax.set_title('AUC-ROC — todos los experimentos y modelos', fontweight='bold', pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=25)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ml_01_heatmap_auc.png', bbox_inches='tight')
plt.show()
print('✓ ml_01_heatmap_auc.png')

In [ ]:
# Barras AUC-ROC por escenario y modelo — clínico vs máquina
for label_tag in ['clinico', 'maquina']:
    subset = results_df[results_df['label_system'] == label_tag]
    if subset.empty: continue

    pivot_s = subset.pivot(index='scenario', columns='model', values='auc_roc_mean')
    pivot_e = subset.pivot(index='scenario', columns='model', values='auc_roc_std')

    fig, ax = plt.subplots(figsize=(13, 5))
    x = np.arange(len(pivot_s))
    w = 0.13
    colors = ['#6B9EC7', '#E05C4B', '#5A8F59', '#F4A236', '#8E6BBF', '#888']

    for i, (model, color) in enumerate(zip(pivot_s.columns, colors)):
        ax.bar(x + i*w, pivot_s[model], w, label=model, color=color, alpha=0.85,
               yerr=pivot_e[model], capsize=3, error_kw={'linewidth': 1})

    ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='Azar')
    ax.set_xticks(x + w * 2.5)
    ax.set_xticklabels(pivot_s.index, rotation=15, ha='right')
    ax.set_ylabel('AUC-ROC (media ± std, CV-5)')
    ax.set_ylim(0, 1.15)
    ax.set_title(f'AUC-ROC por escenario — etiquetado {label_tag}', fontweight='bold')
    ax.legend(loc='upper right', fontsize=8, ncol=2)

    plt.tight_layout()
    fname = f'ml_02_auc_roc_{label_tag}.png'
    plt.savefig(FIGURES_DIR / fname, bbox_inches='tight')
    plt.show()
    print(f'✓ {fname}')

In [ ]:
# Curvas ROC (mejor modelo por escenario, etiquetado clínico)
best_clinico = best_per_exp[best_per_exp['label_system'] == 'clinico'].reset_index(drop=True)
n_plots = len(best_clinico)
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

cv_plot = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for idx, row in best_clinico.iterrows():
    if idx >= 4: break
    ax = axes[idx]

    scen  = next(s for s in SCENARIOS if s[0] == row['scenario'])
    _, pos_cls, neg_cls = scen
    X_r, y_r, _ = prepare_binary_dataset(df, 'label_clinico', pos_cls, neg_cls)

    pipeline = get_base_models()[row['model']]
    tprs, aucs_fold = [], []
    mean_fpr = np.linspace(0, 1, 100)

    for tr_idx, te_idx in cv_plot.split(X_r, y_r):
        pipeline.fit(X_r[tr_idx], y_r[tr_idx])
        y_prob = pipeline.predict_proba(X_r[te_idx])[:, 1]
        fpr, tpr, _ = roc_curve(y_r[te_idx], y_prob)
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        aucs_fold.append(auc(fpr, tpr))

    mean_tpr     = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc     = np.mean(aucs_fold)
    std_auc      = np.std(aucs_fold)
    std_tpr      = np.std(tprs, axis=0)

    ax.plot(mean_fpr, mean_tpr, '#4B9CD3', lw=2,
            label=f'AUC = {mean_auc:.3f} ± {std_auc:.3f}')
    ax.fill_between(mean_fpr, np.clip(mean_tpr-std_tpr, 0, 1),
                    np.clip(mean_tpr+std_tpr, 0, 1), alpha=0.15, color='#4B9CD3')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.set_title(f'{row["scenario"]}\n({row["model"]})', fontweight='bold', fontsize=9)
    ax.legend(loc='lower right', fontsize=8)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])

fig.suptitle('Curvas ROC — mejor modelo por escenario (clínico)', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ml_03_roc_curves.png', bbox_inches='tight')
plt.show()
print('✓ ml_03_roc_curves.png')

In [ ]:
# Impacto del reetiquetado: ΔAUC (máquina − clínico)
clin = results_df[results_df['label_system']=='clinico'].groupby(['scenario','model'])['auc_roc_mean'].mean()
maq  = results_df[results_df['label_system']=='maquina'].groupby(['scenario','model'])['auc_roc_mean'].mean()

delta = (maq - clin).unstack('model').fillna(0)

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(delta, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-0.2, vmax=0.2,
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': 'ΔAUC (máquina − clínico)'})
ax.set_title('Impacto del reetiquetado S4VM en AUC-ROC', fontweight='bold', pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=20)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ml_04_delta_auc_relabeling.png', bbox_inches='tight')
plt.show()
print('✓ ml_04_delta_auc_relabeling.png')

## 7. Análisis de explicabilidad — SHAP

SHAP (SHapley Additive exPlanations) cuantifica la contribución de cada biomarcador
acústico a la predicción del modelo. Se aplica al mejor modelo del escenario
con mayor AUC-ROC (Lundberg y Lee, 2017).

In [ ]:
# Seleccionar el mejor experimento global para SHAP
best_global = results_df.loc[results_df['auc_roc_mean'].idxmax()]
print(f'Mejor experimento global: {best_global["experiment"]}  →  {best_global["model"]}')
print(f'AUC-ROC: {best_global["auc_roc_mean"]:.3f}')

scen_name   = best_global['scenario']
model_name  = best_global['model']
label_sys   = f"label_{best_global['label_system']}"

scen = next(s for s in SCENARIOS if s[0] == scen_name)
_, pos_cls, neg_cls = scen
X_shap, y_shap, _ = prepare_binary_dataset(df, label_sys, pos_cls, neg_cls)

# Entrenar el modelo sobre todo el dataset (para SHAP)
pipeline_shap = get_base_models()[model_name]
pipeline_shap.fit(X_shap, y_shap)

# Extraer el scaler y el clasificador del pipeline
scaler_shap = pipeline_shap.named_steps['scaler']
clf_shap    = pipeline_shap.named_steps['clf']
X_scaled    = scaler_shap.transform(X_shap)

print(f'✓ Modelo entrenado sobre {len(X_shap)} muestras para SHAP')

In [ ]:
# Calcular valores SHAP
if model_name in ('LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost', 'SVM'):
    explainer = shap.TreeExplainer(clf_shap)
    shap_values = explainer.shap_values(X_scaled)

    # Normalizar a array 2D (n_samples, n_features)
    # TreeExplainer puede devolver:
    #   - list de 2 arrays [neg_class, pos_class]  → tomar índice 1
    #   - array 3D (n_samples, n_features, n_classes) → tomar [:, :, 1]
    #   - array 2D (n_samples, n_features)           → ya correcto
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]          # lista → clase positiva
    elif shap_values.ndim == 3:
        shap_vals = shap_values[:, :, 1]   # 3D → clase positiva
    else:
        shap_vals = shap_values             # 2D → ya correcto
else:
    # SVM / LogisticRegression → KernelExplainer (más lento, muestra reducida)
    background = shap.kmeans(X_scaled, 10)
    explainer  = shap.KernelExplainer(clf_shap.predict_proba, background)
    shap_raw   = explainer.shap_values(X_scaled[:30])
    # KernelExplainer con predict_proba devuelve lista [neg, pos]
    shap_vals  = shap_raw[1] if isinstance(shap_raw, list) else shap_raw

# Garantizar que shap_vals es estrictamente 2D antes de calcular importancia
shap_vals = np.array(shap_vals)
if shap_vals.ndim == 3:
    shap_vals = shap_vals[:, :, 1]
assert shap_vals.ndim == 2, f'shap_vals inesperadamente {shap_vals.ndim}D tras normalización'
assert shap_vals.shape[1] == len(FEATURE_COLS), (
    f'Dimensión de features no coincide: {shap_vals.shape[1]} vs {len(FEATURE_COLS)}'
)

# DataFrame de importancia media |SHAP| — ahora siempre 1D
shap_importance = pd.DataFrame({
    'feature':   FEATURE_COLS,
    'mean_shap': np.abs(shap_vals).mean(axis=0),   # axis=0 sobre (n_samples, n_features) → 1D
}).sort_values('mean_shap', ascending=False)

shap_importance.to_csv(PROCESSED_DIR / 'shap_importance.csv', index=False)
print(f'shap_vals shape: {shap_vals.shape}  (esperado: ({len(X_shap)}, {len(FEATURE_COLS)}))')
print(f'Top 10 features SHAP:')
print(shap_importance.head(10).to_string(index=False))

In [ ]:
# Plot: Top 25 features por importancia SHAP media
top25 = shap_importance.head(25)

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(range(len(top25)), top25['mean_shap'],
        color='#5B8DB8', edgecolor='white')
ax.set_yticks(range(len(top25)))
ax.set_yticklabels(top25['feature'], fontsize=8)
ax.set_xlabel('Importancia SHAP media (|valor|)')
ax.set_title(
    f'Top 25 biomarcadores — {model_name}\n'
    f'{scen_name} ({label_sys.replace("label_", "")})',
    fontweight='bold',
)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ml_05_shap_importance.png', bbox_inches='tight')
plt.show()
print('✓ ml_05_shap_importance.png')

In [ ]:
# SHAP Beeswarm plot (solo para modelos basados en árboles)
if model_name in ('RandomForest', 'XGBoost', 'LightGBM', 'CatBoost'):
    top20_feat = shap_importance.head(20)['feature'].tolist()
    feat_idx   = [FEATURE_COLS.index(f) for f in top20_feat]

    shap_exp = shap.Explanation(
        values=shap_vals[:, feat_idx],
        base_values=explainer.expected_value if not isinstance(explainer.expected_value, list)
                    else explainer.expected_value[1],
        data=X_scaled[:, feat_idx],
        feature_names=top20_feat,
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    shap.plots.beeswarm(shap_exp, max_display=20, show=False)
    plt.title(f'SHAP Beeswarm — {model_name} / {scen_name}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'ml_06_shap_beeswarm.png', bbox_inches='tight')
    plt.show()
    print('✓ ml_06_shap_beeswarm.png')
else:
    print(f'Beeswarm no disponible para {model_name}. Ver ml_05_shap_importance.png.')

## 8. Comparativa Optuna vs base

In [ ]:
if not optuna_df.empty:
    print('=== MEJORA OPTUNA vs BASE ===')
    print(optuna_df[['experiment', 'model', 'auc_base', 'auc_optuna', 'delta_auc']]
          .round(4).to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(optuna_df))
    ax.bar(x - 0.2, optuna_df['auc_base'],    0.35, label='Base',   color='#A0B8CF')
    ax.bar(x + 0.2, optuna_df['auc_optuna'],  0.35, label='Optuna', color='#4B9CD3')
    ax.set_xticks(x)
    ax.set_xticklabels(optuna_df['experiment'], rotation=20, ha='right', fontsize=8)
    ax.set_ylabel('AUC-ROC')
    ax.set_ylim(0.4, 1.05)
    ax.set_title('AUC-ROC: modelo base vs optimizado con Optuna', fontweight='bold')
    ax.legend()
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'ml_07_optuna_vs_base.png', bbox_inches='tight')
    plt.show()
    print('✓ ml_07_optuna_vs_base.png')

## 9. Resumen final

In [ ]:
print('=' * 70)
print('RESUMEN DE EXPERIMENTOS DE APRENDIZAJE AUTOMÁTICO')
print('=' * 70)

print(f'\n[Configuración]')
print(f'  Sujetos         : {len(df)}')
print(f'  Features        : {len(FEATURE_COLS)}')
print(f'  Modelos         : LogisticRegression, RandomForest, XGBoost, LightGBM, CatBoost, SVM')
print(f'  Validación      : Stratified K-Fold  k={N_FOLDS}')
print(f'  Optimización    : Optuna TPE ({N_OPTUNA_TRIALS} trials/modelo)')
print(f'  Explicabilidad  : SHAP TreeExplainer / KernelExplainer')
print(f'  Total runs base : {len(results_df)}')

best_g = results_df.loc[results_df['auc_roc_mean'].idxmax()]
print(f'\n[Mejor resultado global]')
print(f'  Experimento : {best_g["experiment"]}')
print(f'  Modelo      : {best_g["model"]}')
print(f'  AUC-ROC     : {best_g["auc_roc_mean"]:.3f} ± {best_g["auc_roc_std"]:.3f}')
print(f'  F1-macro    : {best_g["f1_macro_mean"]:.3f}')
print(f'  Recall      : {best_g["recall_mean"]:.3f}')
print(f'  Especific.  : {best_g["specificity_mean"]:.3f}')

print(f'\n[Mejor modelo por experimento]')
for _, r in best_per_exp.iterrows():
    print(f'  {r["experiment"]:35s}  {r["model"]:20s}  AUC={r["auc_roc_mean"]:.3f} ± {r["auc_roc_std"]:.3f}')

print(f'\n[Top 5 biomarcadores SHAP]')
for _, r in shap_importance.head(5).iterrows():
    print(f'  {r["feature"]:25s}  |SHAP|={r["mean_shap"]:.5f}')

print(f'\n[Figuras generadas]')
for f in sorted(FIGURES_DIR.glob('ml_*.png')):
    print(f'  {f.name}')

print(f'\n[Archivos de resultados]')
for f in sorted(PROCESSED_DIR.glob('ml_*.csv')):
    print(f'  {f.name}')
for f in sorted(PROCESSED_DIR.glob('shap_*.csv')):
    print(f'  {f.name}')

print('\n' + '=' * 70)
print('→ Siguiente: notebooks/llm_reports.ipynb')
print('=' * 70)

---
## Checklist para la memoria (Capítulos 3, 4 y 5)

### Capítulo 3 — Metodología
| Elemento | Código |
|----------|--------|
| 6 modelos evaluados | `get_base_models()` |
| Justificación `RobustScaler` | EDA: 44 features con outliers |
| Justificación `class_weight='balanced'` | EDA: desbalanceo de clases |
| Validación CV estratificada k=5 | `StratifiedKFold` |
| Optuna búsqueda bayesiana | `get_optuna_objective()` |
| SHAP para explicabilidad | `shap.TreeExplainer` |
| MLflow para trazabilidad | `mlflow.start_run()` |

### Capítulo 4 — Resultados (poblar con los outputs de este notebook)
| Elemento | Figura/Archivo |
|----------|----------------|
| Heatmap AUC-ROC | `ml_01_heatmap_auc.png` |
| Barras AUC por escenario | `ml_02_auc_roc_*.png` |
| Curvas ROC | `ml_03_roc_curves.png` |
| Impacto reetiquetado | `ml_04_delta_auc_relabeling.png` |
| SHAP importancia | `ml_05_shap_importance.png` |
| SHAP beeswarm | `ml_06_shap_beeswarm.png` |
| Optuna vs base | `ml_07_optuna_vs_base.png` |
| Tabla resultados CSV | `ml_results_base.csv`, `ml_results_optuna.csv` |

**Siguiente paso:** `notebooks/llm_reports.ipynb`